Historical plotting notebook. Start Jupyter from the repository root; use the experiment READMEs and command-line runners to generate inputs first. Cells retain exploratory plotting variants; Run All is not the reproduction entry point.

In [ ]:
from pathlib import Path
import os, sys
while not Path("pyproject.toml").exists():
    if Path.cwd() == Path.cwd().parent: raise RuntimeError("Start Jupyter inside this repository")
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
from torchvision import transforms
from torchvision.datasets import MNIST, CIFAR10
from torch.utils.data import DataLoader
from src.models.mlp import PlainNN
from src.analysis.gi_calculation import compute_GI

In [ ]:

# Create output directory for experiment 5
output_dir = "exp2_plots"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Set global plotting style parameters for consistency
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 16

In [ ]:




def compute_gi_along_rsvs_all_at_once(model, sample, V, max_alpha=3.0, step_size=0.01, rsv_scale=56.0, device='cuda:1'):
    """
    Compute GIs along ALL RSV directions in a single batch, using the model's own prediction
    """
    model.eval()
    model = model.to(device)
    
    # Ensure sample has correct dimensions [C, H, W]
    if sample.dim() == 4 and sample.shape[0] == 1:
        sample = sample.squeeze(0)
    
    # Move sample to device
    sample = sample.to(device)
    
    # Get original shape for reshaping RSVs
    if sample.dim() == 3:
        C, H, W = sample.shape
    else:
        H, W = sample.shape[-2:]
        C = 1 if sample.dim() == 2 else sample.shape[0]
    
    original_shape = (C, H, W)
    
    # Get the model's prediction for this sample
    with torch.no_grad():
        outputs = model(sample.unsqueeze(0))
        pred_class = torch.argmax(outputs, dim=1).item()
    
    # Extract and prepare ALL RSVs at once
    rsv_tensors = []
    for i in range(V.shape[1]):
        rsv = V[:, i]
        rsv_norm = np.linalg.norm(rsv)
        
        if rsv_norm > 0:
            # Scale the RSV by rsv_scale as requested
            rsv = rsv / rsv_norm * rsv_scale
            
            # Reshape to match input dimensions
            rsv_reshaped = rsv.reshape(original_shape)
            rsv_tensor = torch.tensor(rsv_reshaped, dtype=torch.float32)
            rsv_tensors.append(rsv_tensor)
        else:
            # Add a zero tensor for consistency
            rsv_tensors.append(torch.zeros(original_shape, dtype=torch.float32))
    
    # Stack the tensors to create a batch
    rsv_tensor_batch = torch.stack(rsv_tensors).to(device)
    
    # Create a batch of sample tensors for the reference point
    sample_tensor = sample.unsqueeze(0).expand(len(rsv_tensor_batch), -1, -1, -1)
    
    # Perturbed samples (sample + RSV direction)
    perturbed_samples = sample_tensor + rsv_tensor_batch
    
    # Calculate GIs using the model's own prediction as the reference class
    pos_gi, neg_gi = compute_GI(
        model=model,
        x1=sample_tensor,
        x2=perturbed_samples,
        direction_normalized=False,  # Already normalized above
        correct_class=pred_class,  # Using the model's prediction instead of true class
        step_size=step_size,
        max_eps=max_alpha,
        device=device
    )
    
    # Process the results
    positive_gi = np.array(pos_gi)
    negative_gi = np.array(neg_gi)
    total_gi = positive_gi + negative_gi
    
    return positive_gi, negative_gi, total_gi#, pred_class  # Also return the predicted class


# Function to load the model
def load_model(checkpoint_path, hidden_dim1, hidden_dim2, input_dim=784, output_dim=10, device='cuda:1'):
    # Import the model class (using the PlainNN definition you provided)
    # Define the class inline to avoid import issues
    class PlainNN(torch.nn.Module):
        def __init__(self, input_dim=28*28, hidden_dim1=256, hidden_dim2=256, output_dim=10):
            super(PlainNN, self).__init__()
            self.fc1 = torch.nn.Linear(input_dim, hidden_dim1)
            self.fc2 = torch.nn.Linear(hidden_dim1, hidden_dim2)
            self.fc3 = torch.nn.Linear(hidden_dim2, output_dim)
            self.relu = torch.nn.ReLU()
            self.dropout = torch.nn.Dropout(p=0)

        def forward(self, x):
            x = x.view(x.size(0), -1)
            x = self.dropout(self.relu(self.fc1(x)))
            x = self.dropout(self.relu(self.fc2(x)))
            x = self.fc3(x)
            return x

        def extract_features(self, x):
            features = []
            x = x.view(x.size(0), -1)
            features.append(x)
            x = self.relu(self.fc1(x))
            features.append(x)
            x = self.relu(self.fc2(x))
            features.append(x)
            x = self.fc3(x)
            features.append(x)
            return features
    
    model = PlainNN(input_dim=input_dim, hidden_dim1=hidden_dim1, 
                   hidden_dim2=hidden_dim2, output_dim=output_dim).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model

# Modified function to compute Jacobian of the model at a given input
def compute_jacobian(model, x, output_layer_idx=-2):
    """Compute Jacobian of penultimate layer features w.r.t. input."""
    x.requires_grad_(True)
    
    # Use the extract_features method that's available in the model
    features = model.extract_features(x)
    
    # Get the features from the penultimate layer (before the final classification layer)
    # The index -2 corresponds to the features before the final layer
    penultimate_features = features[output_layer_idx]
    
    # Get feature dimensions
    batch_size, feature_dim = penultimate_features.shape
    input_dim = x.numel()
    
    # Initialize Jacobian matrix
    jacobian = torch.zeros(feature_dim, input_dim, device=x.device)
    
    # Compute Jacobian one feature at a time
    for i in range(feature_dim):
        model.zero_grad()
        
        # Set all gradients to zero except for the current feature
        if i > 0:
            x.grad.zero_()
        
        # Set up a "fake" loss that's just the value of this feature
        penultimate_features[:, i].backward(retain_graph=(i < feature_dim-1))
        
        # Store the gradient
        jacobian[i] = x.grad.view(-1)
    
    # Reset requires_grad
    x.requires_grad_(False)
    
    return jacobian

# Function to compute SVD
def compute_svd(jacobian):
    """Compute SVD of the Jacobian."""
    J_np = jacobian.detach().cpu().numpy()
    U, S, Vt = np.linalg.svd(J_np, full_matrices=True)
    V = Vt.T  # So V[:, i] is the i-th right singular vector
    
    return U, S, V

# Function to compute GI along a direction
def compute_gi_along_direction(model, x, direction, class_id, max_eps=3.0, step_size=0.01, device='cuda:1'):
    """
    Compute GI along a given direction.
    Returns the maximum epsilon for which the prediction remains the same.
    """
    # Normalize the direction
    direction = direction / np.linalg.norm(direction)
    direction_tensor = torch.tensor(direction, dtype=torch.float32, device=device)
    
    # Get the original model prediction
    with torch.no_grad():
        output = model(x)
        original_pred = torch.argmax(output, dim=1).item()
    
    # If the original prediction doesn't match the expected class, return 0
    if original_pred != class_id:
        return 0.0
    
    # Check increasing steps along the direction
    current_eps = 0.0
    while current_eps <= max_eps:
        current_eps += step_size
        
        # Compute the perturbed input
        x_perturbed = x + current_eps * direction_tensor.reshape(x.shape)
        
        # Get the prediction for the perturbed input
        with torch.no_grad():
            output = model(x_perturbed)
            pred = torch.argmax(output, dim=1).item()
        
        # If the prediction changes, return the previous epsilon
        if pred != original_pred:
            return current_eps - step_size
    
    # If we reach here, the prediction remained the same for all tested epsilons
    return max_eps

# Function to get average GIs across singular value directions
# Function to get average GIs across singular value directions
def compute_average_gis_by_sv_index(model, samples, labels, max_eps=3.0, step_size=0.01, device='cuda:1'):
    """
    Compute average GIs along singular vectors for a batch of samples
    using the all-at-once approach.
    """
    num_samples = len(samples)
    
    # We'll gather all total_gi values for each sample
    all_sample_gis = []
    
    for i, (sample, label) in enumerate(tqdm(zip(samples, labels), total=num_samples, desc="Computing GIs")):
        # Reshape for proper processing
        if sample.dim() == 1:  # For flattened samples, reshape to 1x28x28
            sample = sample.reshape(1, 28, 28)
        
        # Move to device
        sample = sample.to(device)
        
        # Compute Jacobian
        jacobian = compute_jacobian(model, sample.unsqueeze(0))
        
        # Compute SVD
        _, S, V = compute_svd(jacobian)
        
        # Compute GIs along all RSV directions at once
        pos_gi, neg_gi, total_gi = compute_gi_along_rsvs_all_at_once(
            model=model,
            sample=sample,
            V=V,
            # true_class=label,
            max_alpha=max_eps,
            step_size=step_size,
            rsv_scale=56.0,  # Use the fixed scaling factor
            device=device
        )
        
        all_sample_gis.append(total_gi)
    
    # Average the GIs across all samples
    if all_sample_gis:
        # Make sure all arrays have the same length
        min_length = min(len(gis) for gis in all_sample_gis)
        all_sample_gis = [gis[:min_length] for gis in all_sample_gis]
        
        # Convert to numpy array and compute mean along samples axis
        avg_gis_by_sv = np.mean(all_sample_gis, axis=0)
    else:
        avg_gis_by_sv = np.array([])
    
    return avg_gis_by_sv

# Main function to analyze and visualize GI expansion
def analyze_gi_expansion(early_checkpoint, later_checkpoint, hidden_dim1, hidden_dim2, 
                        data_path, output_dir="exp3_plots", title_info="", 
                        early_label="Early Training", later_label="Later Training",
                        num_samples_per_class=2, device='cuda:1', figsize=(10, 6)):
    """
    Analyze GI expansion between models at different training stages.
    
    Args:
        early_checkpoint: Path to early training checkpoint
        later_checkpoint: Path to later training checkpoint
        hidden_dim1: Hidden dimension 1
        hidden_dim2: Hidden dimension 2
        data_path: Path to dataset
        output_dir: Directory to save output plots
        title_info: Custom title information to display
        early_label: Label for early training model in legend
        later_label: Label for later training model in legend
        num_samples_per_class: Number of samples to analyze per class
        device: Device to run computations on
        figsize: Figure size for plots
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Set up the device
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load models
    early_model = load_model(early_checkpoint, hidden_dim1, hidden_dim2, device=device)
    later_model = load_model(later_checkpoint, hidden_dim1, hidden_dim2, device=device)
    
    # Load dataset
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    train_dataset = MNIST(root=data_path, train=True, transform=transform, download=True)
    
    # Create dataloader for prediction evaluation
    train_loader = DataLoader(train_dataset, batch_size=100, shuffle=False)
    
    # Get predictions from the later model
    trained_predictions = []
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc="Evaluating trained model")):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = later_model(images)
            _, predicted = torch.max(outputs, 1)
            
            for i in range(len(labels)):
                sample_idx = batch_idx * 100 + i
                is_correct = (predicted[i] == labels[i]).item()
                
                if is_correct:  # Only keep correctly classified samples
                    trained_predictions.append({
                        'sample_idx': sample_idx,
                        'true_label': labels[i].item(),
                        'is_correct': is_correct
                    })
    
    # Group correctly classified samples by class
    correct_samples_by_class = {}
    for pred in trained_predictions:
        class_idx = pred['true_label']
        if class_idx not in correct_samples_by_class:
            correct_samples_by_class[class_idx] = []
        correct_samples_by_class[class_idx].append(pred['sample_idx'])
    
    # Select random samples from each class
    selected_samples = []
    selected_labels = []
    
    for class_idx in range(10):  # Assuming 10 classes for MNIST
        if class_idx in correct_samples_by_class and len(correct_samples_by_class[class_idx]) > 0:
            # Get the list of indices for this class
            indices = correct_samples_by_class[class_idx]
            
            # Select min(num_samples_per_class, available samples) random indices
            k = min(num_samples_per_class, len(indices))
            selected_indices = random.sample(indices, k)
            
            for idx in selected_indices:
                # Get the sample and add it to our list
                sample, label = train_dataset[idx]
                # Ensure the sample is flattened
                sample = sample.flatten()
                selected_samples.append(sample)
                selected_labels.append(label)
    
    print(f"Selected {len(selected_samples)} samples for analysis")
    
    # Compute average GIs for early model
    early_avg_gis = compute_average_gis_by_sv_index(
        early_model, selected_samples, selected_labels, max_eps=20.0, device=device
    )

    # Compute average GIs for later model
    later_avg_gis = compute_average_gis_by_sv_index(
        later_model, selected_samples, selected_labels, max_eps=20.0, device=device
    )
    
    # Get x-axis values (sorted singular value indices)
    sv_indices = np.arange(len(early_avg_gis))
    
    # Create smoothed versions using a sliding window median
    def sliding_median(data, window=5):
        smoothed = np.zeros_like(data)
        for i in range(len(data)):
            start = max(0, i - window//2)
            end = min(len(data), i + window//2 + 1)
            smoothed[i] = np.median(data[start:end])
        return smoothed
    
    # Apply smoothing
    window_size = 5  # Adjust window size as needed
    early_smoothed = sliding_median(early_avg_gis, window=window_size)
    later_smoothed = sliding_median(later_avg_gis, window=window_size)
    
    # Use more soothing colors
    early_color = '#5DA5DA'  # sky blue
    later_color = '#60BD68'  # soft green
    
    # Plot the smoothed lines (regular scale)
    plt.figure(figsize=figsize)
    
    # Add model and dataset info to the title
    title = title_info if title_info else "GI Expansion During Training"
    
    # Plot the smoothed lines with more soothing colors
    plt.plot(sv_indices, early_smoothed, color=early_color, linewidth=2.5, 
             label=early_label)
    plt.plot(sv_indices, later_smoothed, color=later_color, linewidth=2.5, 
             label=later_label)
    
    # Add the raw data as semi-transparent points
    plt.scatter(sv_indices, early_avg_gis, alpha=0.2, color=early_color, s=10)
    plt.scatter(sv_indices, later_avg_gis, alpha=0.2, color=later_color, s=10)
    
    # Customize the plot
    plt.xlabel('Singular Value Index (sorted)')
    plt.ylabel('Average Generalization Interval (GI)')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Remove the gray background
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['figure.facecolor'] = 'white'
    
    # Save the plot
    plt.tight_layout()
    out_path = os.path.join(output_dir, 'gi_expansion_comparison_smoothed')
    plt.savefig(f"{out_path}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_path}.pdf", format='pdf', bbox_inches='tight')
    
    # Create log-scale version
    plt.figure(figsize=figsize)
    
    plt.plot(sv_indices, early_smoothed, color=early_color, linewidth=2.5, 
             label=early_label)
    plt.plot(sv_indices, later_smoothed, color=later_color, linewidth=2.5, 
             label=later_label)
    
    # Add the raw data as semi-transparent points
    plt.scatter(sv_indices, early_avg_gis, alpha=0.2, color=early_color, s=10)
    plt.scatter(sv_indices, later_avg_gis, alpha=0.2, color=later_color, s=10)
    
    plt.xlabel('Singular Value Index (sorted)')
    plt.ylabel('Average Generalization Interval (GI) - Log Scale')
    plt.title(title)
    plt.yscale('log')  # Log scale for y-axis
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Remove the gray background
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['figure.facecolor'] = 'white'
    
    plt.tight_layout()
    out_path_log = os.path.join(output_dir, 'gi_expansion_comparison_smoothed_log')
    plt.savefig(f"{out_path_log}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_path_log}.pdf", format='pdf', bbox_inches='tight')
    
    # Show the plots
    plt.show()
    
    # Return the data for further analysis if needed
    return {
        'sv_indices': sv_indices,
        'early_avg_gis': early_avg_gis,
        'later_avg_gis': later_avg_gis,
        'early_smoothed': early_smoothed,
        'later_smoothed': later_smoothed
    }




In [ ]:

# Define paths and parameters
early_checkpoint = "checkpoints/mnist_original_mlp/model_epoch0_batch399.pth"
later_checkpoint = "checkpoints/mnist_original_mlp/model_epoch49_batch399.pth"
data_path = "../data/MNIST-Original"
hidden_dim1 = 1024  # Adjust based on your model
hidden_dim2 = 512  # Adjust based on your model
    
results = analyze_gi_expansion(
    early_checkpoint=early_checkpoint,
    later_checkpoint=later_checkpoint,
    hidden_dim1=hidden_dim1,
    hidden_dim2=hidden_dim2,
    data_path=data_path,
    output_dir="exp2_plots",
    title_info="GI Expansion During Training on MNIST",
    early_label="Early Training (Epoch 1)",
    later_label="Later Training (Epoch 50)",
    num_samples_per_class=5,
    device="cuda:1",
    figsize=(6,3)
)

## Rubbish codes


In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import yaml
import importlib
from torchvision import transforms
from torchvision.datasets import MNIST, CIFAR10
from torch.utils.data import DataLoader

from src.analysis.gi_calculation import compute_GI

def compute_gi_along_rsvs_all_at_once(model, sample, V, max_alpha=3.0, step_size=0.01, rsv_scale=56.0, device='cuda'):
    """
    Compute GIs along ALL RSV directions in a single batch, using the model's own prediction
    """
    model.eval()
    model = model.to(device)
    
    # Ensure sample has correct dimensions [C, H, W]
    if sample.dim() == 4 and sample.shape[0] == 1:
        sample = sample.squeeze(0)
    
    # Move sample to device
    sample = sample.to(device)
    
    # Get original shape for reshaping RSVs
    if sample.dim() == 3:
        C, H, W = sample.shape
    else:
        H, W = sample.shape[-2:]
        C = 1 if sample.dim() == 2 else sample.shape[0]
    
    original_shape = (C, H, W)
    
    # Get the model's prediction for this sample
    with torch.no_grad():
        outputs = model(sample.unsqueeze(0))
        pred_class = torch.argmax(outputs, dim=1).item()
    
    # Extract and prepare ALL RSVs at once
    rsv_tensors = []
    for i in range(V.shape[1]):
        rsv = V[:, i]
        rsv_norm = np.linalg.norm(rsv)
        
        if rsv_norm > 0:
            # Scale the RSV by rsv_scale as requested
            rsv = rsv / rsv_norm * rsv_scale
            
            # Reshape to match input dimensions
            rsv_reshaped = rsv.reshape(original_shape)
            rsv_tensor = torch.tensor(rsv_reshaped, dtype=torch.float32)
            rsv_tensors.append(rsv_tensor)
        else:
            # Add a zero tensor for consistency
            rsv_tensors.append(torch.zeros(original_shape, dtype=torch.float32))
    
    # Stack the tensors to create a batch
    rsv_tensor_batch = torch.stack(rsv_tensors).to(device)
    
    # Create a batch of sample tensors for the reference point
    sample_tensor = sample.unsqueeze(0).expand(len(rsv_tensor_batch), -1, -1, -1)
    
    # Perturbed samples (sample + RSV direction)
    perturbed_samples = sample_tensor + rsv_tensor_batch
    
    # Calculate GIs using the model's own prediction as the reference class
    pos_gi, neg_gi = compute_GI(
        model=model,
        x1=sample_tensor,
        x2=perturbed_samples,
        direction_normalized=False,  # Already normalized above
        correct_class=pred_class,  # Using the model's prediction instead of true class
        step_size=step_size,
        max_eps=max_alpha,
        device=device
    )
    
    # Process the results
    positive_gi = np.array(pos_gi)
    negative_gi = np.array(neg_gi)
    total_gi = positive_gi + negative_gi
    
    return positive_gi, negative_gi, total_gi

def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

def get_model(model_config):
    """Dynamically import and instantiate the model based on config."""
    model_name = model_config['name']
    model_args = model_config.get('args', {})
    
    # Import the module containing the model
    module_path = model_config.get('module', 'src.models')
    
    # Use the 'file' field from config if available, otherwise fallback to lowercased model name
    model_file = model_config.get('file', model_name.lower())
    
    # Import the module
    module = importlib.import_module(f"{module_path}.{model_file}")
    
    # Get the model class and instantiate it
    model_class = getattr(module, model_name)
    return model_class(**model_args)

def compute_jacobian(model, x, output_layer_idx=-2):
    """
    Compute Jacobian of penultimate layer features w.r.t. input.
    
    Args:
        model: Neural network model
        x: Input tensor
        output_layer_idx: Index of the layer to extract features from
    """
    x.requires_grad_(True)
    
    # Different model types have different ways to extract features
    model_type = model.__class__.__name__
    
    # Try to use extract_features method if available
    if hasattr(model, 'extract_features'):
        features = model.extract_features(x)
        penultimate_features = features[output_layer_idx]
    # Try to use get_representation method if available
    elif hasattr(model, 'get_representation'):
        penultimate_features = model.get_representation(x)
    # For CNN models
    elif 'CNN' in model_type:
        # Implement CNN feature extraction if needed
        # This is a simple implementation - adjust based on your CNN architecture
        batch_size = x.size(0)
        feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
        penultimate_features = feature_extractor(x).view(batch_size, -1)
    # For ViT models
    elif 'ViT' in model_type:
        # For ViT models, use the embedding output if available
        if hasattr(model, 'get_embedding'):
            penultimate_features = model.get_embedding(x)
        else:
            # Extract all but the classification head
            feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
            penultimate_features = feature_extractor(x)
    # Default fallback - assume the model has a simple structure with a classifier at the end
    else:
        # Remove the last layer (classifier) and get the output
        feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
        penultimate_features = feature_extractor(x)
        if len(penultimate_features.shape) > 2:  # If features are not flattened
            penultimate_features = penultimate_features.view(x.size(0), -1)
    
    # Get feature dimensions
    batch_size, feature_dim = penultimate_features.shape
    input_dim = x.numel()
    
    # Initialize Jacobian matrix
    jacobian = torch.zeros(feature_dim, input_dim, device=x.device)
    
    # Compute Jacobian one feature at a time
    for i in range(feature_dim):
        model.zero_grad()
        
        # Set all gradients to zero except for the current feature
        if i > 0:
            x.grad.zero_()
        
        # Set up a "fake" loss that's just the value of this feature
        penultimate_features[:, i].backward(retain_graph=(i < feature_dim-1))
        
        # Store the gradient
        jacobian[i] = x.grad.view(-1)
    
    # Reset requires_grad
    x.requires_grad_(False)
    
    return jacobian

def compute_svd(jacobian):
    """Compute SVD of the Jacobian."""
    J_np = jacobian.detach().cpu().numpy()
    U, S, Vt = np.linalg.svd(J_np, full_matrices=True)
    V = Vt.T  # So V[:, i] is the i-th right singular vector
    
    return U, S, V

def compute_average_gis_by_sv_index(model, samples, labels, max_eps=3.0, step_size=0.01, device='cuda'):
    """
    Compute average GIs along singular vectors for a batch of samples
    using the all-at-once approach.
    """
    num_samples = len(samples)
    
    # We'll gather all total_gi values for each sample
    all_sample_gis = []
    
    for i, (sample, label) in enumerate(tqdm(zip(samples, labels), total=num_samples, desc="Computing GIs")):
        # Ensure sample is correctly shaped for the model
        if sample.dim() != 4:
            # Check if it's a flattened image
            if sample.dim() == 1:
                # Determine if it's MNIST (1 channel) or CIFAR (3 channels)
                if sample.size(0) == 784:  # MNIST 28x28=784
                    sample = sample.reshape(1, 28, 28)
                elif sample.size(0) == 3072:  # CIFAR 32x32x3=3072
                    sample = sample.reshape(3, 32, 32)
                else:
                    # Use best guess based on size
                    if sample.size(0) % 3 == 0:  # Likely color image
                        size = int(np.sqrt(sample.size(0) / 3))
                        sample = sample.reshape(3, size, size)
                    else:  # Likely grayscale
                        size = int(np.sqrt(sample.size(0)))
                        sample = sample.reshape(1, size, size)
                        
        # Add batch dimension if needed
        if sample.dim() == 3:
            sample = sample.unsqueeze(0)
            
        # Move to device
        sample = sample.to(device)
        
        # Compute Jacobian
        jacobian = compute_jacobian(model, sample)
        
        # Compute SVD
        _, S, V = compute_svd(jacobian)
        
        # Compute GIs along all RSV directions at once
        pos_gi, neg_gi, total_gi = compute_gi_along_rsvs_all_at_once(
            model=model,
            sample=sample,
            V=V,
            max_alpha=max_eps,
            step_size=step_size,
            rsv_scale=56.0,  # Use the fixed scaling factor
            device=device
        )
        
        all_sample_gis.append(total_gi)
    
    # Average the GIs across all samples
    if all_sample_gis:
        # Make sure all arrays have the same length
        min_length = min(len(gis) for gis in all_sample_gis)
        all_sample_gis = [gis[:min_length] for gis in all_sample_gis]
        
        # Convert to numpy array and compute mean along samples axis
        avg_gis_by_sv = np.mean(all_sample_gis, axis=0)
    else:
        avg_gis_by_sv = np.array([])
    
    return avg_gis_by_sv

def get_dataset(config):
    """Get dataset based on config."""
    dataset_name = config['dataset']['name'].lower()
    data_dir = config['dataset']['data_dir']
    
    # Get normalization parameters from config
    transform_config = config['dataset'].get('transform', {})
    mean = transform_config.get('normalize', {}).get('mean', (0.5,))
    std = transform_config.get('normalize', {}).get('std', (0.5,))
    
    # Define transformations
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    # Load the appropriate dataset
    if dataset_name == 'mnist':
        train_dataset = MNIST(root='../data/MNIST-Original', train=True, transform=transform, download=True)
    elif dataset_name == 'cifar10':
        train_dataset = CIFAR10(root='../data/CIFAR10', train=True, transform=transform, download=True)
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")
    
    return train_dataset, dataset_name

def analyze_gi_expansion(early_checkpoint, later_checkpoint, config_path, 
                         num_samples_per_class=2, device='cuda'):
    """
    Analyze GI expansion between models at different training stages.
    
    Args:
        early_checkpoint: Path to checkpoint from early training
        later_checkpoint: Path to checkpoint from later training
        config_path: Path to model configuration file
        num_samples_per_class: Number of samples per class to analyze
        device: Device to run on
    """
    # Set up the device
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Extract epoch info from checkpoint paths
    early_epoch = int(early_checkpoint.split('epoch')[1].split('_')[0])
    early_batch = int(early_checkpoint.split('batch')[1].split('.')[0])
    later_epoch = int(later_checkpoint.split('epoch')[1].split('_')[0])
    later_batch = int(later_checkpoint.split('batch')[1].split('.')[0])
    
    # Load configuration
    config = load_config(config_path)
    
    # Load models using the configuration
    early_model = get_model(config['model']).to(device)
    later_model = get_model(config['model']).to(device)
    
    # Load weights
    early_model.load_state_dict(torch.load(early_checkpoint, map_location=device))
    later_model.load_state_dict(torch.load(later_checkpoint, map_location=device))
    
    # Set models to evaluation mode
    early_model.eval()
    later_model.eval()
    
    # Model architecture info
    model_name = config['model']['name']
    model_info = f"{model_name}"
    
    # Additional model details based on model type
    if 'PlainNN' in model_name or 'MLP' in model_name:
        hidden_dim1 = config['model']['args'].get('hidden_dim1', 'N/A')
        hidden_dim2 = config['model']['args'].get('hidden_dim2', 'N/A')
        model_info += f" ({hidden_dim1}, {hidden_dim2})"
    elif 'CNN' in model_name:
        if 'hidden_channels1' in config['model']['args']:
            hidden_channels1 = config['model']['args'].get('hidden_channels1', 'N/A')
            hidden_channels2 = config['model']['args'].get('hidden_channels2', 'N/A')
            model_info += f" ({hidden_channels1}, {hidden_channels2})"
    elif 'ViT' in model_name:
        embed_dim = config['model']['args'].get('embed_dim', 'N/A')
        depth = config['model']['args'].get('depth', 'N/A')
        num_heads = config['model']['args'].get('num_heads', 'N/A')
        model_info += f" (dim={embed_dim}, depth={depth}, heads={num_heads})"
    
    # Load dataset
    train_dataset, dataset_name = get_dataset(config)
    dataset_info = dataset_name.upper()
    
    # Create dataloader for prediction evaluation
    train_loader = DataLoader(train_dataset, batch_size=100, shuffle=False)
    
    # Get predictions from the later model
    trained_predictions = []
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc="Evaluating trained model")):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = later_model(images)
            _, predicted = torch.max(outputs, 1)
            
            for i in range(len(labels)):
                sample_idx = batch_idx * 100 + i
                is_correct = (predicted[i] == labels[i]).item()
                
                if is_correct:  # Only keep correctly classified samples
                    trained_predictions.append({
                        'sample_idx': sample_idx,
                        'true_label': labels[i].item(),
                        'is_correct': is_correct
                    })
    
    # Group correctly classified samples by class
    correct_samples_by_class = {}
    for pred in trained_predictions:
        class_idx = pred['true_label']
        if class_idx not in correct_samples_by_class:
            correct_samples_by_class[class_idx] = []
        correct_samples_by_class[class_idx].append(pred['sample_idx'])
    
    # Determine number of classes from dataset
    num_classes = 10  # Default for MNIST and CIFAR-10
    if 'num_classes' in config['dataset']:
        num_classes = config['dataset']['num_classes']
    
    # Select random samples from each class
    selected_samples = []
    selected_labels = []
    
    for class_idx in range(num_classes):
        if class_idx in correct_samples_by_class and len(correct_samples_by_class[class_idx]) > 0:
            # Get the list of indices for this class
            indices = correct_samples_by_class[class_idx]
            
            # Select min(num_samples_per_class, available samples) random indices
            k = min(num_samples_per_class, len(indices))
            selected_indices = random.sample(indices, k)
            
            for idx in selected_indices:
                # Get the sample and add it to our list
                sample, label = train_dataset[idx]
                selected_samples.append(sample)
                selected_labels.append(label)
    
    print(f"Selected {len(selected_samples)} samples for analysis")
    
    # Compute average GIs for early model
    early_avg_gis = compute_average_gis_by_sv_index(
        early_model, selected_samples, selected_labels, max_eps=20.0, device=device
    )

    # Compute average GIs for later model
    later_avg_gis = compute_average_gis_by_sv_index(
        later_model, selected_samples, selected_labels, max_eps=20.0, device=device
    )
    
    # Get x-axis values (sorted singular value indices)
    sv_indices = np.arange(len(early_avg_gis))
    
    # Create smoothed versions using a sliding window median
    def sliding_median(data, window=5):
        smoothed = np.zeros_like(data)
        for i in range(len(data)):
            start = max(0, i - window//2)
            end = min(len(data), i + window//2 + 1)
            smoothed[i] = np.median(data[start:end])
        return smoothed
    
    # Apply smoothing
    window_size = 5  # Adjust window size as needed
    early_smoothed = sliding_median(early_avg_gis, window=window_size)
    later_smoothed = sliding_median(later_avg_gis, window=window_size)
    
    # Use more soothing colors
    early_color = '#5DA5DA'  # sky blue
    later_color = '#60BD68'  # soft green
    
    # Set up figure with a clean style (no gray background)
    plt.style.use('default')  # Reset to default style
    plt.figure(figsize=(12, 7), facecolor='white')  # White background
    
    # Add model and dataset info to the title
    title = f"GI Expansion During Training on {dataset_info}\nModel: {model_info}"
    
    # Plot the smoothed lines with more soothing colors
    plt.plot(sv_indices, early_smoothed, color=early_color, linewidth=2.5, 
             label=f'Early Training (Epoch {early_epoch}, Batch {early_batch})')
    plt.plot(sv_indices, later_smoothed, color=later_color, linewidth=2.5, 
             label=f'Later Training (Epoch {later_epoch}, Batch {later_batch})')
    
    # Add the raw data as semi-transparent points
    plt.scatter(sv_indices, early_avg_gis, alpha=0.2, color=early_color, s=10)
    plt.scatter(sv_indices, later_avg_gis, alpha=0.2, color=later_color, s=10)
    
    # Customize the plot
    plt.xlabel('Singular Value Index (sorted)') #, fontsize=12)
    plt.ylabel('Average Generalization Interval (GI))') #, fontsize=12)
    plt.title(title) #, fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Add a text box with additional information
    info_text = f"Dataset: {dataset_info}\nModel: {model_info}\nSamples: {len(selected_samples)}"
    plt.figtext(0.02, 0.02, info_text,
                bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))
    
    # Generate output filename based on model and dataset
    output_filename = f"gi_expansion_{dataset_name}_{model_name.lower()}"
    
    # Save the plot
    plt.tight_layout()
    plt.savefig(f'{output_filename}_smoothed.png', dpi=300, facecolor='white')  # White background
    
    # Create log-scale version
    plt.figure(figsize=(12, 7), facecolor='white')  # White background
    
    plt.plot(sv_indices, early_smoothed, color=early_color, linewidth=2.5, 
             label=f'Early Training (Epoch {early_epoch}, Batch {early_batch})')
    plt.plot(sv_indices, later_smoothed, color=later_color, linewidth=2.5, 
             label=f'Later Training (Epoch {later_epoch}, Batch {later_batch})')
    
    # Add the raw data as semi-transparent points
    plt.scatter(sv_indices, early_avg_gis, alpha=0.2, color=early_color, s=10)
    plt.scatter(sv_indices, later_avg_gis, alpha=0.2, color=later_color, s=10)
    
    plt.xlabel('Singular Value Index (sorted)') #, fontsize=12)
    plt.ylabel('Average Generalization Interval (GI) - Log Scale') #, fontsize=12)
    plt.title(title) #, fontsize=14)
    plt.yscale('log')  # Log scale for y-axis
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Add the same info text
    plt.figtext(0.02, 0.02, info_text, 
                bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))
    
    plt.tight_layout()
    plt.savefig(f'{output_filename}_smoothed_log.png', dpi=300, facecolor='white')  # White background
    
    # Return the data for further analysis if needed
    return {
        'sv_indices': sv_indices,
        'early_avg_gis': early_avg_gis,
        'later_avg_gis': later_avg_gis,
        'early_smoothed': early_smoothed,
        'later_smoothed': later_smoothed,
        'model_info': model_info,
        'dataset_info': dataset_info
    }


In [ ]:
# Define paths and parameters
early_checkpoint = "checkpoints/mnist_original_mlp/model_epoch0_batch399.pth"
later_checkpoint = "checkpoints/mnist_original_mlp/model_epoch49_batch399.pth"
config_path = "configs/training/mnist_mlp.yaml"

# Run the analysis
results = analyze_gi_expansion(
    early_checkpoint=early_checkpoint,
    later_checkpoint=later_checkpoint,
    config_path=config_path,
    num_samples_per_class=2,  # 2 samples per class
    device="cuda:1"  # Use "cpu" if no GPU available
)

In [ ]:
early_checkpoint = "checkpoints/mnist_original_cnn/model_epoch0_batch374.pth"
later_checkpoint = "checkpoints/mnist_original_cnn/model_epoch29_batch374.pth"
config_path = "configs/training/mnist_cnn.yaml"

# Run the analysis
results = analyze_gi_expansion(
    early_checkpoint=early_checkpoint,
    later_checkpoint=later_checkpoint,
    config_path=config_path,
    num_samples_per_class=2,  # 2 samples per class
    device="cuda"  # Use "cpu" if no GPU available
)

In [ ]:
early_checkpoint = "checkpoints/mnist_original_cnn/model_epoch0_batch374.pth"
later_checkpoint = "checkpoints/mnist_original_cnn/model_epoch14_batch374.pth"
config_path = "configs/training/mnist_vit.yaml"

# Run the analysis
results = analyze_gi_expansion(
    early_checkpoint=early_checkpoint,
    later_checkpoint=later_checkpoint,
    config_path=config_path,
    num_samples_per_class=2,  # 2 samples per class
    device="cuda"  # Use "cpu" if no GPU available
)